# ODSC 2026: Deploying a Secure Autonomous Agent

We're building an agent that autonomously monitors teacher licensure compliance requirements across US states and districts — detecting changes and alerting you the moment something updates.

| Part | What you build | Complexity |
| --- | --- | --- |
| **Part 1** | Environment setup and first compliance check | Setup |
| **Part 2** | Custom Claw — 150 lines of Python, full monitor | Build it yourself |
| **Part 3** | Add Telegram notifications to Custom Claw | Extend it |
| **Part 4** | Switch to OpenClaw — same logic, production framework | Use a framework |
| **Part 5** | Observability with Opik — trace every agent decision | Production ready |

**Prerequisites:** Docker running, `.env` filled in from `example.env`.

---
## Part 1: Environment Setup

In [ ]:
import subprocess
import os
from dotenv import load_dotenv

load_dotenv(override=True)

# Check required keys
OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY", "")
BRAVE_API_KEY   = os.getenv("BRAVE_API_KEY", "")

assert OPENAI_API_KEY and not OPENAI_API_KEY.startswith("your-"), "Set OPENAI_API_KEY in .env"
assert BRAVE_API_KEY  and not BRAVE_API_KEY.startswith("your-"),  "Set BRAVE_API_KEY in .env"

# Check Docker
docker_ok = subprocess.run("docker info".split(), capture_output=True).returncode == 0
assert docker_ok, "Docker is not running. Start Docker Desktop and try again."

print("✅ OpenAI key set")
print("✅ Brave key set")
print("✅ Docker running")
print("\nReady to go.")

---
## Part 2: Custom Claw — Build It Yourself

Before we bring in a framework, let's see the core logic: ~150 lines of Python that fetches state DOE pages, hashes the content, and detects changes.

Read `custom-claw/src/monitor.py` — that's the entire agent.

### What it does
1. Fetches each URL in `compliance_urls.py`
2. Strips HTML, hashes the text content with SHA-256
3. Compares against the stored hash in SQLite
4. Records a change if the hash differs
5. Sends a Telegram alert (Part 3)

In [ ]:
import shutil

# Auto-detect Docker sudo requirement
SUDO = ""
result = subprocess.run("docker info".split(), capture_output=True)
if result.returncode != 0:
    result = subprocess.run("sudo docker info".split(), capture_output=True)
    if result.returncode == 0:
        SUDO = "sudo "

DOCKER_COMPOSE = (
    f"{SUDO}docker compose"
    if subprocess.run(f"{SUDO}docker compose version".split(), capture_output=True).returncode == 0
    else f"{SUDO}docker-compose"
)

# Sync .env into custom-claw directory
shutil.copy(".env", "custom-claw/.env")

print(f"Using: {DOCKER_COMPOSE}")
print("Building Custom Claw container...")
!{DOCKER_COMPOSE} -f custom-claw/docker-compose.yml build
print("\n✅ Custom Claw built.")

In [ ]:
# Run the first compliance check — this captures baseline snapshots
print("Running first compliance check (capturing baselines)...\n")
!{DOCKER_COMPOSE} -f custom-claw/docker-compose.yml run --rm custom-claw

In [ ]:
# Check a specific state
!{DOCKER_COMPOSE} -f custom-claw/docker-compose.yml run --rm custom-claw python monitor.py Ohio

In [ ]:
# View the change log (SQLite)
import sqlite3
import os

DB_PATH = "data/compliance.db"
if os.path.exists(DB_PATH):
    conn = sqlite3.connect(DB_PATH)
    
    print("=== Snapshots ===")
    rows = conn.execute("SELECT state, subject, checked_at FROM snapshots ORDER BY checked_at DESC LIMIT 15").fetchall()
    for r in rows:
        print(f"  {r[2][:16]}  {r[0]} — {r[1]}")
    
    print("\n=== Changes Detected ===")
    rows = conn.execute("SELECT state, subject, url, detected_at FROM changes ORDER BY detected_at DESC").fetchall()
    if rows:
        for r in rows:
            print(f"  🚨 {r[3][:16]}  {r[0]} — {r[1]}")
            print(f"     {r[2]}")
    else:
        print("  No changes yet (baselines just captured).")
    
    conn.close()
else:
    print("Database not found — run the compliance check first (cell above).")

---
## Part 3: Telegram Notifications

Add `TELEGRAM_BOT_TOKEN` and `TELEGRAM_CHAT_ID` to your `.env` to receive alerts on your phone when compliance changes are detected.

See the [Telegram Setup](#telegram-setup) section in the README for how to create a bot and get your chat ID.

Once set, re-run the check from Part 2 — any detected change will be sent as a Telegram message.

In [ ]:
load_dotenv(override=True)

BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN", "")
CHAT_ID   = os.getenv("TELEGRAM_CHAT_ID", "")

if not BOT_TOKEN or not CHAT_ID:
    print("⚠️  TELEGRAM_BOT_TOKEN or TELEGRAM_CHAT_ID not set in .env")
    print("   Follow the Telegram setup in README.md, then re-run this cell.")
else:
    import requests
    resp = requests.post(
        f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage",
        json={"chat_id": CHAT_ID, "text": "✅ Compliance monitor connected. You'll receive alerts here when licensure requirements change."},
        timeout=10
    )
    if resp.ok:
        print("✅ Telegram connected — check your phone!")
    else:
        print(f"❌ Telegram error: {resp.text}")

---
## Part 4: Switch to OpenClaw

Custom Claw works, but it's a single Python script with no scheduling, no dashboard, no memory, and no multi-channel support. 

[OpenClaw](https://github.com/openclaw/openclaw) is a production agent framework that wraps the same logic with:
- Persistent memory and conversation context
- Web dashboard for monitoring
- Built-in scheduling
- Telegram, Slack, Discord, WhatsApp out of the box
- 50+ built-in skills
- Skills defined as Markdown — no Python required

The `check-compliance-updates` skill in `openclaw-agent/skills/` is the OpenClaw equivalent of `monitor.py`.

In [ ]:
import shutil
import time

OPENCLAW_COMPOSE = f"{DOCKER_COMPOSE} -f openclaw-agent/docker-compose.yml"

# Sync .env
shutil.copy(".env", "openclaw-agent/.env")

print("Starting OpenClaw gateway...")
!{OPENCLAW_COMPOSE} up -d openclaw-gateway

for i in range(15):
    health = subprocess.run(
        f"{OPENCLAW_COMPOSE} exec openclaw-gateway curl -sf http://localhost:18789/healthz".split(),
        capture_output=True
    )
    if health.returncode == 0:
        print("✅ Gateway healthy!")
        break
    time.sleep(3)
    print(f"   Waiting... ({i+1})")

In [ ]:
# Configure the agent
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs config set agents.defaults.model openai/gpt-4o
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs config set gateway.bind lan
print("\n✅ Agent configured.")

In [ ]:
# Ask the agent to run a compliance check
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs agent -m "Check for any teacher licensure compliance updates in Ohio and Texas."

In [ ]:
# Connect Telegram to OpenClaw (get pairing code)
print("Adding Telegram channel to OpenClaw...")

TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN", "")
assert TELEGRAM_BOT_TOKEN, "Set TELEGRAM_BOT_TOKEN in .env first"

!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs channels add --channel telegram --token {TELEGRAM_BOT_TOKEN}

print("\n✅ Telegram channel added.")
print("Send the pairing code shown above to your bot on Telegram to complete setup.")

---
## Part 5: Observability with Opik

OpenClaw is now running, but we have no visibility into *how* it's making decisions — which tools it called, how long they took, what it cost.

Run `opik_observability.ipynb` to install the Opik plugin and start capturing full traces for every agent turn.

Then run `opik_evaluation.ipynb` to batch-evaluate your agent's response quality with LLM-as-a-judge metrics.